# MuCoCo RQ3 Experiment Results Aggregation for Model Confidence

This notebook is used to aggregate the results for MuCoCo RQ3 experiments. The results should be stored in MuCoCo_results/MuCoCo_experiment_results/model_output_confidence in the project root folder. The final aggregated results from this notebook are used in tables XI (mpact of varying mode confidence on Inconsistency and Accuracy).

In [22]:
import os
import sys
import pandas as pd

In [23]:
current_dir = os.getcwd()
proj_dir = os.path.abspath(os.path.join(current_dir, ".."))
main_dir = os.path.abspath(os.path.join(proj_dir, ".."))
sys.path.append(proj_dir)
sys.path.append(main_dir)

In [24]:
from utility.data_log_functions import DataLogHelper

In [25]:
def check_confidence(model_folder_dir: str, model_name: str, task: str):
    csv_files = [file for file in os.listdir(model_folder_dir) if file.endswith('.csv')]
    thresholds = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]
    benchmarks = ["HumanEval", "CruxEval", "CodeMMLU"]

    for benchmark in benchmarks:
        benchmark_files = [file for file in csv_files if benchmark in file and 'ensemble' not in file]

        for file in benchmark_files:
            file_path = os.path.join(model_folder_dir, file)
            data = pd.read_csv(file_path)

            # Make a copy of the base columns
            benchmark_df = data.loc[:, ['task_id', 'geometric', 'failure_type']].copy()

            # For each threshold, create a new column for classified failure type
            for threshold in thresholds:
                new_col = f'failure_type_{threshold}'
                classified_values = []

                for conf, failure in zip(data['geometric'], data['failure_type']):
                    if isinstance(failure, float) or (isinstance(failure, str) and "AssertionError" in failure and "Mutation" not in failure):
                        if conf >= threshold:
                            classified_values.append(failure)  # correct (no error)
                        else:
                            classified_values.append("Confidence_lower_than_threshold") 
                    else:
                        classified_values.append(failure)

                benchmark_df[new_col] = classified_values

            # Save as new CSV
            out_path = os.path.join(proj_dir, f"MuCoCo_experiment_results/model_output_confidence/{task}/{model_name}")
            os.makedirs(name = out_path, exist_ok=True)
            benchmark_df.to_csv(os.path.join(out_path,f"{file.split('.csv')[0]}_confidence.csv"), index=False)


In [26]:
main_res_dir = os.path.join(proj_dir, "MuCoCo_experiment_results/")

for task in ["input_prediction", "mcq_inconsistency"]:
    target_dir = os.path.join(main_res_dir, task)
    for file in os.listdir(target_dir):
        if any(m.lower() in file.lower() for m in ["Gemma", "Qwen", "Llama"]):
            model_folder_path = os.path.join(target_dir, file)
            check_confidence(model_folder_path, file, task)

In [34]:
conf_dir = os.path.join(main_res_dir, "model_output_confidence")
thresholds = ["", 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]

res_df = pd.DataFrame()
acc_df = pd.DataFrame()
inconsistency_dict = {}
accuracy_dict = {}

LLM_EXECUTION_ERROR = "LLM Execution Error"
LLM_CORRECTNESS_ERROR = "LLM Correctness Error"
LLM_ANSWER_CORRECT = "LLM Answer Correct"

# Confidence threshold for these 2 tasks only.
for task in ['input_prediction', 'mcq_inconsistency']:
    res_dir = os.path.join(conf_dir, task)

    # iterating through each model (gemma, llama, qwen)
    for models in os.listdir(res_dir):
        model_inc_dict = inconsistency_dict.get(models, {})
        model_acc_dict = accuracy_dict.get(models, {})
        if models == ".DS_Store":
            continue
        model_res_path = os.path.join(res_dir, models)

        # iterating through each benchmarks (humaneval, codemmlu, cruxeval)
        for benchmark in ["HumanEval", "CodeMMLU", "CruxEval"]:

            # obtaining the file names of the various csv logs
            csv_logs = [log for log in os.listdir(model_res_path) if benchmark in log and log.endswith('.csv')]

            # skip if no valid logs
            if len(csv_logs) == 0:
                continue
            
            # obtaining the no mutation log to compare to
            no_mut = [log for log in csv_logs if "no_mutation" in log][0]
            no_mut_data = pd.read_csv(os.path.join(model_res_path, no_mut))

            # removing the no mutation log from csv_logs
            csv_logs.pop(csv_logs.index(no_mut))

            # iterating csv logs through each threshold ("", 0.5,... etc)
            for threshold in thresholds:

                total_comparisons = 0
                total_inconsistencies = 0

                total_answered = 0
                total_correct = 0
                
                # if statement for the column header names
                if threshold != "":
                    col_name = f"failure_type_{threshold}"
                else: 
                    col_name = f"failure_type"

                # obtain the no mutation failure types
                no_mut_failure_type = no_mut_data.loc[:, col_name]

                # iterating through each failure_type in the no mutation csv log
                for no_mut_failure in no_mut_failure_type:
                    failure_dict = DataLogHelper.verify_failure_type_relevance(no_mut_failure)

                    correct = failure_dict[LLM_ANSWER_CORRECT]
                    incorrect = failure_dict[LLM_CORRECTNESS_ERROR]

                    # Not a valid error for confidence testing. Invalid errors are ignored
                    if correct == False and incorrect == False:
                        continue

                    # if llm answer is correct
                    elif correct:
                        total_correct += 1

                    total_answered += 1


                # iterating through other logs
                for mut_log in sorted(csv_logs):
                    
                    # loading the data
                    mut_data = pd.read_csv(os.path.join(model_res_path, mut_log))

                    # obtaining the failure types
                    mut_failure_type = mut_data.loc[:, col_name]

                    no_mut_temp = no_mut_data.copy()
                    mut_temp = mut_data.copy()

                    no_mut_temp, mut_temp = DataLogHelper.standardize_two_df(no_mut_temp, mut_temp)

                    # iterating through each pair of failure type in the no mutation log and mutation log
                    for idx, (no_mut_failure , mut_failure) in enumerate(zip(no_mut_temp.loc[:, col_name], mut_temp.loc[:, col_name])):
                        
                        res1_result = DataLogHelper.verify_failure_type_relevance(no_mut_failure)
                        res2_result = DataLogHelper.verify_failure_type_relevance(mut_failure)

                        # determining the outcome of the llm output: either correct or incorrect
                        res1_correct = res1_result[LLM_ANSWER_CORRECT]
                        res2_correct = res2_result[LLM_ANSWER_CORRECT]
                        res1_wrong = res1_result[LLM_CORRECTNESS_ERROR]
                        res2_wrong = res2_result[LLM_CORRECTNESS_ERROR]
                        res1_invalid = res1_result[LLM_EXECUTION_ERROR]
                        res2_invalid = res2_result[LLM_EXECUTION_ERROR]

                        if res2_correct or res2_wrong:
                            total_answered += 1

                        if res1_correct and res2_correct:
                            total_correct += 1
                        
                        elif res1_invalid and res2_invalid:
                            pass

                        elif(res1_correct and (res2_wrong or res2_invalid)) or (res2_correct and (res1_wrong or res1_invalid)):
                            total_inconsistencies += 1
                        
                        elif (res1_wrong and res2_invalid) or (res2_wrong and res1_invalid):
                            total_inconsistencies += 1

                        elif (res1_wrong and res2_wrong):
                            # need to fix here
                            inconsistency_exists = mut_temp.iloc[idx]["model_output"] != no_mut_temp.iloc[idx]["model_output"]
                            total_inconsistencies += 1 if inconsistency_exists else 0
                        
                        else:
                            continue
                        
                        total_comparisons += 1


                sub_inc_dict = model_inc_dict.get(threshold, {"comparisons": 0, "inconsistencies": 0})
                sub_inc_dict['comparisons'] += total_comparisons
                sub_inc_dict['inconsistencies'] += total_inconsistencies
                model_inc_dict[threshold] = sub_inc_dict

                sub_acc_dict = model_acc_dict.get(threshold, {})
                sub_acc_dict['correct'] = total_correct
                sub_acc_dict['answered'] = total_answered
                model_acc_dict[threshold] = sub_acc_dict


                try:
                    res_df.loc[f"{threshold}", f"{models}_{benchmark}_{task}"] = f"{total_inconsistencies}/{total_comparisons} = {round(total_inconsistencies*100/total_comparisons, 2)}"
                except ZeroDivisionError:
                    res_df.loc[f"{threshold}", f"{models}_{benchmark}_{task}"] = f"{total_inconsistencies}/{total_comparisons} = 0"
                
                try:
                    acc_df.loc[f"{threshold}", f"{models}_{benchmark}_{task}"] = f"{total_correct}/{total_answered} = {round(total_correct*100/total_answered, 2)}"
                except ZeroDivisionError:
                    res_df.loc[f"{threshold}", f"{models}_{benchmark}_{task}"] = f"{total_correct}/{total_answered} = 0"
            

        
        inconsistency_dict[models] = model_inc_dict
        accuracy_dict[models] = model_acc_dict


new_df_index = [
    "gemma-3-12b-it_HumanEval_input_prediction",
    "gemma-3-12b-it_CruxEval_input_prediction",
    "gemma-3-12b-it_CodeMMLU_mcq_inconsistency",
    "Qwen2.5-Coder-14B-Instruct_HumanEval_input_prediction",
    "Qwen2.5-Coder-14B-Instruct_CruxEval_input_prediction",
    "Qwen2.5-Coder-14B-Instruct_CodeMMLU_mcq_inconsistency",
    "LLama-3.1-8B_HumanEval_input_prediction",
    "LLama-3.1-8B_CruxEval_input_prediction",
    "LLama-3.1-8B_CodeMMLU_mcq_inconsistency"
]


res_df = res_df.reindex(new_df_index, axis=1)
acc_df = acc_df.reindex(new_df_index, axis = 1)

In [35]:
from collections import defaultdict

def merge_by_confidence(data):
    merged = defaultdict(lambda: {"inconsistencies": 0, "comparisons": 0})
    
    for model, conf_dict in data.items():
        for conf, stats in conf_dict.items():
            merged[conf]["inconsistencies"] += stats.get("inconsistencies", 0)
            merged[conf]["comparisons"] += stats.get("comparisons", 0)
    
    return dict(merged)

# Example usage
merged_data1 = merge_by_confidence(inconsistency_dict)

## Inconsistency of models (gemma, qwen, llama) at each confidence threshold

In [36]:
for threshold, res_dict in merged_data1.items():
    inconsistencies = res_dict['inconsistencies']
    comparisons = res_dict['comparisons']

    res_df.loc[f"{threshold}", f"Aggregated"] = f"{inconsistencies}/{comparisons} = {round(inconsistencies*100/comparisons, 2)}"

print(res_df.to_string())

     gemma-3-12b-it_HumanEval_input_prediction gemma-3-12b-it_CruxEval_input_prediction gemma-3-12b-it_CodeMMLU_mcq_inconsistency Qwen2.5-Coder-14B-Instruct_HumanEval_input_prediction Qwen2.5-Coder-14B-Instruct_CruxEval_input_prediction Qwen2.5-Coder-14B-Instruct_CodeMMLU_mcq_inconsistency LLama-3.1-8B_HumanEval_input_prediction LLama-3.1-8B_CruxEval_input_prediction LLama-3.1-8B_CodeMMLU_mcq_inconsistency         Aggregated
                               299/5620 = 5.32                          316/3355 = 9.42                           221/699 = 31.62                                       186/6106 = 3.05                                      214/3355 = 6.38                                       186/699 = 26.61                         527/5620 = 9.38                        156/3355 = 4.65                         239/704 = 33.95  2344/29513 = 7.94
0.5                            299/5620 = 5.32                          316/3355 = 9.42                           221/699 = 31.62             

## Accuracy of models (gemma, qwen, llama) at each confidence threshold

In [37]:
def merge_by_confidence(data):
    merged = defaultdict(lambda: {"answered": 0, "correct": 0})
    
    for model, conf_dict in data.items():
        for conf, stats in conf_dict.items():
            merged[conf]["answered"] += stats.get("answered", 0)
            merged[conf]["correct"] += stats.get("correct", 0)
    
    return dict(merged)


merged_data2 = merge_by_confidence(accuracy_dict)


for threshold, res_dict in merged_data2.items():
    answered = res_dict['answered']
    correct = res_dict['correct']
    # print(answered)

    acc_df.loc[f"{threshold}", f"Aggregated"] = f"{correct}/{answered} = {round(correct*100/answered, 2)}"

print(acc_df.to_string())

     gemma-3-12b-it_HumanEval_input_prediction gemma-3-12b-it_CruxEval_input_prediction gemma-3-12b-it_CodeMMLU_mcq_inconsistency Qwen2.5-Coder-14B-Instruct_HumanEval_input_prediction Qwen2.5-Coder-14B-Instruct_CruxEval_input_prediction Qwen2.5-Coder-14B-Instruct_CodeMMLU_mcq_inconsistency LLama-3.1-8B_HumanEval_input_prediction LLama-3.1-8B_CruxEval_input_prediction LLama-3.1-8B_CodeMMLU_mcq_inconsistency         Aggregated
                             5321/6615 = 80.44                        3035/4155 = 73.04                           408/841 = 48.51                                     6181/7101 = 87.04                                    3387/4155 = 81.52                                       518/841 = 61.59                       2869/6615 = 43.37                      3793/4155 = 91.29                          272/837 = 32.5  1198/2519 = 47.56
0.5                          5321/6615 = 80.44                        3035/4155 = 73.04                           408/841 = 48.51             

In [49]:
indexes = acc_df.index
columns = acc_df.columns
new_df = pd.DataFrame()
for idx in indexes:
    for col in columns:
        acc_val = acc_df.loc[idx, col]
        inc_val = res_df.loc[idx, col]

        acc = acc_val.split('=')[-1].strip()
        inc = inc_val.split('=')[-1].strip()

        new_df.loc[idx, col] = f"{inc} ({acc})"

print(new_df.to_string())


     gemma-3-12b-it_HumanEval_input_prediction gemma-3-12b-it_CruxEval_input_prediction gemma-3-12b-it_CodeMMLU_mcq_inconsistency Qwen2.5-Coder-14B-Instruct_HumanEval_input_prediction Qwen2.5-Coder-14B-Instruct_CruxEval_input_prediction Qwen2.5-Coder-14B-Instruct_CodeMMLU_mcq_inconsistency LLama-3.1-8B_HumanEval_input_prediction LLama-3.1-8B_CruxEval_input_prediction LLama-3.1-8B_CodeMMLU_mcq_inconsistency    Aggregated
                                  5.32 (80.44)                             9.42 (73.04)                             31.62 (48.51)                                          3.05 (87.04)                                         6.38 (81.52)                                         26.61 (61.59)                            9.38 (43.37)                           4.65 (91.29)                            33.95 (32.5)  7.94 (47.56)
0.5                               5.32 (80.44)                             9.42 (73.04)                             31.62 (48.51)                       